## 🧪 Experiments

In [ ]:
# Imports
from   dotenv   import load_dotenv
import pandas   as pd
import datetime
import traceback
import signal
import psutil
import json
import time
import os
import gc

# Custom Imports
import sys
sys.path.append('../')
import AppUtils 	  
import LLMUtils 
from AnalysisUtils import AnalysisUtils, ObfuscationDetectionAnalysisUtils

##### Initialization

In [ ]:
print("⚡ START: {} ⚡".format(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
initTime = datetime.datetime.now()

In [ ]:
# Paths 
TMP_PATH           = "../../0_Data/TMP"

# Create TMP Folder 
if not os.path.exists(TMP_PATH):
	os.makedirs(TMP_PATH)
	print("--- 🆕 Folder created : {}".format(TMP_PATH))
else:
	print("--- 📂 TMP Folder ready : {}".format(TMP_PATH))

In [ ]:
# GPU Test
import torch

# Check if CUDA is available
print("--- 🖥️ CUDA available           : {}".format(torch.cuda.is_available()))

# Show the number of GPUs available
numGpus = torch.cuda.device_count()
print("--- 🖥️ Number of available GPUs : {}".format(numGpus))

# Display the names of the GPUs
for i in range(numGpus):
	print("--- 🖥️ GPU {}: {}".format(i, torch.cuda.get_device_name(i)))

In [ ]:
# Load .env Info
load_dotenv()

#### 📥 1] Load Data

In [ ]:
DATA_PATH = os.path.abspath(os.path.join(os.getcwd(), "./test.csv"))

# Load CSV 
appsDf = pd.read_csv(DATA_PATH)
print("--- #️⃣ Number of Apps : {}".format(len(appsDf)))

# Show first 3 rows
appsDf.head(3)

### 🧪 2] LLM Analysis 

In [ ]:
# PAYING
MODEL = "gpt-4o-mini"

# FREE
# MODEL = "gpt-oss:20b"
# MODEL = "deepseek-r1:32b"
# MODEL = "gemma3:27b"
# MODEL = "qwen3:30b"
# MODEL = "llama3.1:8b"
# MODEL = "phi3:14b"

# Context Window Size
CONTEXT_WINDOW_SIZE = 128000

# Check and Init
print("\n--- ⭕ LLM: Init & Check")
print("--- ⭐ Model: {}".format(MODEL))

# PAYING
if LLMUtils.isOpenAiModel(MODEL):
	llmInterface = LLMUtils.OpenAiInterface(model=MODEL,contextWindow=CONTEXT_WINDOW_SIZE)
elif MODEL.lower().startswith("gemini"):
	llmInterface = LLMUtils.GeminiInterface(model=MODEL,contextWindow=CONTEXT_WINDOW_SIZE)
else:
	llmInterface = LLMUtils.OllamaInterface(model=MODEL,contextWindow=CONTEXT_WINDOW_SIZE)

# Send a test request to check if the LLM is working
print("--- 🔸 LLM Response : {}".format(llmInterface.sendRequest("Ping!")))

##### Parameters

In [ ]:
# To computer the minimum number of classes for statistically significant random sample
CONFIDENCE_LEVEL = 99
ERROR_MARGIN     = 5

# Threshold for obfuscation detection
OBFUSCATION_THRESHOLD = 0.5

# Filtering [None | "system" | "tp" | "both" | "pkgNameOnly"]
FILTERING = "both"

# Sampling
RANDOM_SEED    = 777

# LLM Robustness
NUM_ITERATIONS = 3
MAX_RETRIES    = 3

# Logging
SILENT_MODE = True

In [ ]:
# Prompt Selection
PROMPTS_PATH  = "../prompt.yaml"
PROMPT_ID     = "ObfuscationDetectionV1"

# Prompt Config & Helpers 
prompts        = AnalysisUtils.loadPrompts(PROMPTS_PATH)
promptInfo     = AnalysisUtils.getPromptById(prompts, PROMPT_ID)
promptTemplate = promptInfo["promptTemplate"]

print("--- 📝 Prompt ID          : {}".format(promptInfo["promptID"]))
print("--- 📝 Prompt Description : {}".format(promptInfo["promptDescription"]))
print("--- 📝 Prompt Template    : {}".format(promptTemplate))

In [ ]:
# Results & Resume
OUTPUT_PATH_JSON = "./results/json/obfuscationDetection_{}_Results.json".format(MODEL.replace(":", "_").replace("/", "_"))
OUTPUT_PATH_CSV  = "./results/csv/obfuscationDetection_{}_Results.csv".format(MODEL.replace(":", "_").replace("/", "_"))

# Only analyze apps that have not been analyzed yet (resume capability)
obfuscationDetectionResults = AnalysisUtils.loadExistingResults(OUTPUT_PATH_JSON)
completedSha256Set       = {result["sha256"] for result in obfuscationDetectionResults}
pendingAppsDf            = appsDf[~appsDf["sha256"].isin(completedSha256Set)].reset_index(drop = True)

print("\n--- ⭕ Checking for existing results...")
if len(obfuscationDetectionResults) > 0:
	print("--- ☑️ File Found!")
	print("--- 🔄️ Current Progress : {}/{}".format(len(obfuscationDetectionResults), len(appsDf)))
else:
	print("--- 🆕 No existing results file found! --> Starting fresh analysis...")

In [ ]:
# Basic App Analysis
for appIdx, appRow in pendingAppsDf.iterrows():
	# Get data from CSV
	sha256    = appRow["sha256"]
	pkgName   = appRow["pkgName"]
	app       = None
	appResult = None

	# Print Message
	print("\n--- 🔑 Analyzing App [{}/{}] : {}".format(appIdx + 1, len(pendingAppsDf), sha256))
	print("--- 📦 pkgName: {}".format(pkgName))

	# Analyze
	try:
		# Create app object 📱
		app = AppUtils.App(sha256, pkgName, TMP_PATH + "/")

		app.downloadAPK()
		app.decompileWithApktool()
		app.collectSmaliClasses()

		# Filter Smali classes before sampling
		print("\n--- ⭕ Filtering Smali Classes before sampling")
		print("--- 🔹 Filtering Strategy : {}".format(FILTERING))
		if FILTERING is None:
			print("--- 🔹 No filtering applied.")
		elif FILTERING == "system":
			app.filterOutSystemLibraries()
		elif FILTERING == "tp":
			app.filterOutThirdPartyLibraries()
		elif FILTERING == "both":
			app.filterOutSystemLibraries()
			app.filterOutThirdPartyLibraries()
		elif FILTERING == "pkgNameOnly":
			app.filterByPkgName()
		else:
			raise ValueError("Unsupported FILTERING mode: {}".format(FILTERING))

		# No smali classes found
		if app.numSmaliClasses == 0:
			print("--- ⚠️ No Smali Classes found.")
			appResult = AnalysisUtils.createResultsObject(
				sha256                    = sha256,
				pkgName                   = pkgName,
				status                    = "NO_SMALI_CLASSES",
				numSmaliClasses           = app.numSmaliClasses,
				numSmaliClassesAnalyzed   = 0,
				pctSmaliClassesObfuscated = 0.0,
				llmFinalLabel             = None
			)
			continue

		# Compute Sample Size
		print("\n--- ⭕ Computing random sample size [confidence={}%, error margin={}%]".format(CONFIDENCE_LEVEL, ERROR_MARGIN))
		numSmaliClassesAnalyzed   = AnalysisUtils.computeRandomSampleSize(app.numSmaliClasses, CONFIDENCE_LEVEL, ERROR_MARGIN)
		print("--- #️⃣ Random Sample Size : {}".format(numSmaliClassesAnalyzed))

		# Test purposes
		numSmaliClassesAnalyzed = 3

		# Get random sample of smali classes
		sampledSmaliClasses  = AnalysisUtils.getRandomSample(app.smaliClasses, numSmaliClassesAnalyzed, RANDOM_SEED)

		# Init and iterate for each smali class
		numSmaliClassesObfuscated = 0
		print("\n--- ⭕ Analyzing Smali Classes with LLM")
		print("--- 🔹 Obfuscation Threshold : {}".format(OBFUSCATION_THRESHOLD))
		print("--- 🔹 Num Iterations        : {}".format(NUM_ITERATIONS))
		print("--- 🔹 Max Retries           : {}\n".format(MAX_RETRIES))

		for smaliIdx, smaliClass in enumerate(sampledSmaliClasses):
			if not SILENT_MODE:
				print("--- 🔸 Checking Smali Class [{}/{}] : {}".format(smaliIdx + 1, numSmaliClassesAnalyzed, smaliClass["className"]))

			# Analyze smali class with LLM and majority vote
			classAnalysis = ObfuscationDetectionAnalysisUtils.analyzeSmaliClassWithMajorityVote(
				llmInterface   = llmInterface,
				smaliClass     = smaliClass,
				promptTemplate = promptTemplate,
				numIterations  = NUM_ITERATIONS,
				maxRetries     = MAX_RETRIES
			)

			# Get majority label and counts
			isObfuscated = classAnalysis["majorityLabel"]
			trueCount    = classAnalysis["trueCount"]
			falseCount   = classAnalysis["falseCount"]
			if not SILENT_MODE:
				print("--- 🏷️ Label Frequency : True={} | False={}".format(trueCount, falseCount))
				print("--- 🏷️ Majority Label  : {}".format(isObfuscated))

			if isObfuscated:
				numSmaliClassesObfuscated += 1
			if not SILENT_MODE:
				print("---"*20)

		pctSmaliClassesObfuscated = numSmaliClassesObfuscated / numSmaliClassesAnalyzed
		llmFinalLabel             = pctSmaliClassesObfuscated >= OBFUSCATION_THRESHOLD

		print("\n--- 🎯 Results for App : {}".format(pkgName))
		print("--- 🔹 N. Obfuscated Smali Classes  : {} / {}".format(numSmaliClassesObfuscated, numSmaliClassesAnalyzed))
		print("--- 🔹 PCT Obfuscated Smali Classes : {:.2f}".format(pctSmaliClassesObfuscated))
		print("--- 🔹 llmFinalLabel [isObfuscated] : {}".format(llmFinalLabel))

		# Append the results
		appResult = AnalysisUtils.createResultsObject(
			sha256                    = sha256,
			pkgName                   = pkgName,
			status                    = "SUCCESS",
			numSmaliClasses           = app.numSmaliClasses,
			numSmaliClassesAnalyzed   = numSmaliClassesAnalyzed,
			pctSmaliClassesObfuscated = pctSmaliClassesObfuscated,
			llmFinalLabel             = llmFinalLabel
		)

	# Handle exceptions and create result object with error status
	except Exception as e:
		print("--- ⚠️ Error while analyzing {} : {}".format(pkgName, e))
		errorTrace = traceback.format_exc().replace("\n", " | ")
		appResult = AnalysisUtils.createResultsObject(
			sha256                    = sha256,
			pkgName                   = pkgName,
			status                    = "ERROR - {}".format(errorTrace),
			numSmaliClasses           = 0 if app is None else app.numSmaliClasses,
			numSmaliClassesAnalyzed   = 0,
			pctSmaliClassesObfuscated = 0.0,
			llmFinalLabel             = None
		)

	finally:
	
		# Save partial results after each app analysis (for resume capability and to avoid losing data in case of crashes)
		if appResult is not None:
			obfuscationDetectionResults.append(appResult)
			AnalysisUtils.saveResults(obfuscationDetectionResults, OUTPUT_PATH_JSON)
			print("\n--- 💾 Partial Report saved : {}".format(OUTPUT_PATH_JSON))

		# Clean up app files to save space
		if app is not None:
			app.deleteAPK()
			app.deleteAll()

	print("+++"*40 + "\n")

##### 💾 Save Results

In [ ]:
# Save Report 
AnalysisUtils.saveResults(obfuscationDetectionResults, OUTPUT_PATH_JSON)
AnalysisUtils.saveResultsAsCsv(obfuscationDetectionResults, OUTPUT_PATH_CSV)
print("--- 💾 JSON Report saved : {}".format(OUTPUT_PATH_JSON))
print("--- 💾 CSV Report saved  : {}".format(OUTPUT_PATH_CSV))

#### 📊 Print some stats/results

In [ ]:
# Compute and print final statistics
resultsStats = ObfuscationDetectionAnalysisUtils.computeResultsStatistics(obfuscationDetectionResults)

print("\n--- 📊 Dataset Statistics")
print("--- 🔹 Total Apps              : {}".format(resultsStats["totalApps"]))
print("--- 🎯 Analyzed Apps           : ")
print("------ 🔹 Success                 : {} [{:.2%}]".format(resultsStats["numSuccess"], resultsStats["pctSuccess"]))
print("------ 🔹 Error                   : {} [{:.2%}]".format(resultsStats["numError"], resultsStats["pctError"]))
print("------ 🔹 Zero Smali Classes      : {} [{:.2%}]".format(resultsStats["numZeroSmaliClasses"], resultsStats["pctZeroSmaliClasses"]))
print("--- 🎯 Obfuscation Detection   : ")
print("------ 🔹 Obfuscated Apps [True]  : {} [{:.2%}]".format(resultsStats["numObfuscatedAppsTrue"], resultsStats["pctObfuscatedAppsTrue"]))
print("------ 🔹 Obfuscated Apps [False] : {} [{:.2%}]".format(resultsStats["numObfuscatedAppsFalse"], resultsStats["pctObfuscatedAppsFalse"]))

##### 🔚 End

In [ ]:
endTime = datetime.datetime.now()
print("\n🔚 --- END:  {} --- 🔚".format(endTime.strftime("%Y-%m-%d %H:%M:%S")))

# Assuming endTime and initTime are datetime objects
totalTime = endTime - initTime
hours     = totalTime.total_seconds() // 3600
minutes   = (totalTime.total_seconds() % 3600) // 60
seconds   = totalTime.total_seconds() % 60
print("⏱️ --- Time: {:02d} hours and {:02d} minutes [{:02d} seconds] --- ⏱️".format(int(hours), int(minutes), int(totalTime.total_seconds())))